# DEX admin: upgrade the canister & add a trading pair

This notebook covers the two operations that require **controller** privileges on the DEX canister:

1. Upgrading the canister to a new WASM
2. Adding a new trading pair

Both calls will reject with `NotController` when run from a non-controller identity.

## Prerequisites

- [`icp` CLI](https://cli.internetcomputer.org/) installed
- Launch this notebook from the **project root** so `--environment staging` resolves the `dex` canister name (defined in `icp.yaml`)
- An identity that is a controller of the DEX canister. The repo convention is the `hsm` identity — adjust `IDENTITY` below if you use a different one.

## Check your identity is a controller

`canister status` lists the controllers. Your principal must appear there, otherwise every update call in this notebook will be rejected.

In [ ]:
IDENTITY = "hsm"   # controller identity; matches the default in the justfile

!icp identity principal --identity "$IDENTITY"
!icp canister status dex --environment staging --identity "$IDENTITY"

---

# Part 1 — Upgrade the canister

Upgrades preserve stable memory: balances, open orders, listed pairs, and the event log all survive. The canister's `post_upgrade` takes an `opt DexArg`; passing `(null)` keeps the current configuration. To change the access mode (General Availability ↔ Restricted), pass a structured `Upgrade` arg as shown below.

## 1.1 Build the WASM

`just build` compiles `dex_canister` to `wasm32-unknown-unknown` in release mode and produces `wasms/dex_canister.wasm.gz`. The icp CLI picks up that artifact automatically via the recipe declared in `icp.yaml`.

In [ ]:
!just build

## 1.2 Pre-upgrade sanity check

Capture a snapshot of observable state to diff against after the upgrade. If state is lost, you will see it here.

In [ ]:
!icp canister call dex get_trading_pairs '()' --environment staging --query

## 1.3 Run the upgrade

Two equivalent paths:

**A. Using the `just` recipe** — defers args to the icp CLI defaults (`(null)`), keeps config unchanged:

```bash
just deploy             # uses IDENTITY=hsm
# just deploy <other>   # override the identity
```

**B. Running the full command explicitly** — gives you control over the upgrade arg:

In [ ]:
UPGRADE_ARGS = "(null)"   # no config change; keeps current mode

!icp canister install dex --mode upgrade --args '$UPGRADE_ARGS' --identity "$IDENTITY" --environment staging -y

## 1.4 Post-upgrade verification

Re-query the state you captured in 1.2. Trading pairs, balances, and open orders should all be unchanged.

In [ ]:
!icp canister call dex get_trading_pairs '()' --environment staging --query

## 1.5 (Optional) Upgrade while changing the access mode

`Mode::GeneralAvailability` lets any principal call update endpoints; `Mode::RestrictedTo` limits update calls to a fixed set of principals (useful for staged rollouts or pausing a misbehaving integration).

To restrict updates to a specific allowlist during the next upgrade, use a structured arg:

In [ ]:
ALLOWED_PRINCIPALS = [
    # "aaaaa-aa",
]

principals_vec = "; ".join(f'principal "{p}"' for p in ALLOWED_PRINCIPALS)
RESTRICT_ARGS = (
    f'(opt variant {{ Upgrade = opt record {{ '
    f'mode = opt variant {{ RestrictedTo = vec {{ {principals_vec} }} }} '
    f'}} }})'
)
print(RESTRICT_ARGS)

# Example — commented out to avoid accidentally restricting staging:
# !icp canister install dex --mode upgrade --args '$RESTRICT_ARGS' --identity "$IDENTITY" --environment staging -y

# To revert to open access on a later upgrade:
GA_ARGS = '(opt variant { Upgrade = opt record { mode = opt variant { GeneralAvailability } } })'
print(GA_ARGS)

---

# Part 2 — Add a trading pair

`add_trading_pair` is an update call restricted to controllers. The request carries both ledger IDs plus the token metadata (`symbol`, `decimals`). The DEX validates the submitted metadata against what's already registered (if either token is already part of another listed pair) — mismatches return `InconsistentTokenMetadata`.

## 2.1 Choose the ledgers

Set the two ledger canister IDs. Base is the asset being bought/sold; quote is the asset prices are denominated in.

In [ ]:
BASE_LEDGER  = "ryjl3-tyaaa-aaaaa-aaaba-cai"   # e.g. ICP ledger
QUOTE_LEDGER = "mxzaz-hqaaa-aaaar-qaada-cai"   # e.g. ckBTC ledger

## 2.2 Fetch ledger metadata

The `symbol` and `decimals` you submit **must** match what each ledger reports via `icrc1_symbol` / `icrc1_decimals`. Mismatches cause either the DEX to reject the call outright (if the token is already registered) or, more insidiously, correctly-matching but wrong metadata that misrepresents the asset.

In [ ]:
!icp canister call "$BASE_LEDGER"  icrc1_symbol   '()' --network ic --query
!icp canister call "$BASE_LEDGER"  icrc1_decimals '()' --network ic --query
!icp canister call "$QUOTE_LEDGER" icrc1_symbol   '()' --network ic --query
!icp canister call "$QUOTE_LEDGER" icrc1_decimals '()' --network ic --query

## 2.3 Choose tick size and lot size

- `tick_size` — the minimum price increment (in quote-token base units per base-token base unit). All order prices must be a positive multiple.
- `lot_size` — the minimum quantity (in base-token base units). All order quantities must be a positive multiple.

Both are `nat64`, both must be > 0, and both are fixed for the lifetime of the pair. Pick values that make sense for the expected price range and per-order granularity.

In [ ]:
BASE_SYMBOL   = "ICP"          # copy from icrc1_symbol output above
BASE_DECIMALS = 8              # copy from icrc1_decimals output above

QUOTE_SYMBOL   = "ckBTC"       # copy from icrc1_symbol output above
QUOTE_DECIMALS = 8             # copy from icrc1_decimals output above

TICK_SIZE = 10                 # minimum price increment (quote base units)
LOT_SIZE  = 1_000_000          # minimum quantity       (base  base units)

## 2.4 Call `add_trading_pair`

Possible errors:

| Error                         | Meaning                                                       |
|-------------------------------|---------------------------------------------------------------|
| `NotController`               | Caller is not a controller of the DEX canister                |
| `BaseEqualsQuote`             | `base` and `quote` resolve to the same ledger                 |
| `InvalidTickSize`             | `tick_size == 0`                                              |
| `InvalidLotSize`              | `lot_size == 0`                                               |
| `TradingPairAlreadyExists`    | The (base, quote) pair is already listed                      |
| `InconsistentTokenMetadata`   | Submitted symbol/decimals differ from previously registered   |

In [ ]:
ADD_PAIR_ARGS = (
    f'(record {{ '
    f'base = record {{ '
    f'id = record {{ ledger_id = principal "{BASE_LEDGER}" }}; '
    f'metadata = record {{ symbol = "{BASE_SYMBOL}"; decimals = {BASE_DECIMALS} : nat8 }} '
    f'}}; '
    f'quote = record {{ '
    f'id = record {{ ledger_id = principal "{QUOTE_LEDGER}" }}; '
    f'metadata = record {{ symbol = "{QUOTE_SYMBOL}"; decimals = {QUOTE_DECIMALS} : nat8 }} '
    f'}}; '
    f'tick_size = {TICK_SIZE} : nat64; '
    f'lot_size = {LOT_SIZE} : nat64 '
    f'}})'
)
print(ADD_PAIR_ARGS)

!icp canister call dex add_trading_pair '$ADD_PAIR_ARGS' --identity "$IDENTITY" --environment staging

## 2.5 Verify the listing

The new pair should appear in the `get_trading_pairs` query:

In [ ]:
!icp canister call dex get_trading_pairs '()' --environment staging --query

## What's next

- The listing is immutable: there is no endpoint to remove a pair or change its tick/lot size once added. If you need different parameters, list a separate pair.
- Every `add_trading_pair` is recorded in the append-only event log — inspect with `get_events` (see `canister/dex.did`).
- See `examples/getting_started.ipynb` for how traders interact with a listed pair (deposit → order → withdraw).